In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from gensim.models import FastText,KeyedVectors

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [2]:
#nltk.download('punkt')
#nltk.download('stopwords')

In [3]:
df1 = pd.read_csv("Mental_health.csv")

In [4]:
df1

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...,...
197199,19671,Cyclical depression Can’t get help because rea...,Depression
197200,2332,Childhood trauma When I was about 9-10 years o...,Depression
197201,19922,little poem or something like that.. I wrote t...,Depression
197202,20599,i think i'm depressed how do I fix it? I'm not...,Depression


In [5]:
df1.drop(columns=['Unnamed: 0'], inplace=True)

In [6]:
df1

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
197199,Cyclical depression Can’t get help because rea...,Depression
197200,Childhood trauma When I was about 9-10 years o...,Depression
197201,little poem or something like that.. I wrote t...,Depression
197202,i think i'm depressed how do I fix it? I'm not...,Depression


In [7]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197204 entries, 0 to 197203
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   statement  197134 non-null  object
 1   status     197204 non-null  object
dtypes: object(2)
memory usage: 3.0+ MB


In [8]:
df1.isna().sum()
df1.dropna(inplace=True)

In [9]:
df1.isna().sum()


statement    0
status       0
dtype: int64

In [10]:
stop_words = set(stopwords.words('english'))

In [11]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return tokens

In [12]:
df1["tokens"] = df1["statement"].apply(clean_text)

In [13]:
df1

,statement,status,tokens
0,oh my gosh,Anxiety,[gosh]
1,"trouble sleeping, confused mind, restless hear...",Anxiety,"[trouble, sleeping, confused, mind, restless, ..."
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,"[wrong, back, dear, forward, doubt, stay, rest..."
3,I've shifted my focus to something else but I'...,Anxiety,"[shifted, focus, something, else, still, worried]"
4,"I'm restless and restless, it's been a month n...",Anxiety,"[restless, restless, month, boy, mean]"
...,...,...,...
197199,Cyclical depression Can’t get help because rea...,Depression,"[cyclical, depression, get, help, reality, kee..."
197200,Childhood trauma When I was about 9-10 years o...,Depression,"[childhood, trauma, years, old, step, uncle, m..."
197201,little poem or something like that.. I wrote t...,Depression,"[little, poem, something, like, wrote, ago, or..."
197202,i think i'm depressed how do I fix it? I'm not...,Depression,"[think, depressed, fix, sure, depressed, cry, ..."


In [15]:
sentences = df1["tokens"].tolist()

In [16]:
df1['status'].value_counts()

status
Depression    49301
Normal        49292
Suicidal      49287
Anxiety       49254
Name: count, dtype: int64

In [17]:
labels = df1['status'].unique()

In [18]:
df = pd.DataFrame()
for label in labels:
    sample_data = df1[df1['status'] == label].sample(n=18000, random_state=42)
    df = pd.concat([df, sample_data])

In [19]:
df = df.reset_index(drop=True)
df['status'].value_counts()

status
Anxiety       18000
Normal        18000
Depression    18000
Suicidal      18000
Name: count, dtype: int64

In [20]:
ft_model = FastText(sentences=sentences,vector_size=150,window=5,min_count=2,workers=4,sg=1,)

In [21]:
def sent_vector(tokens):
    vec = np.zeros(ft_model.vector_size)
    count = 0
    for word in tokens:
        if word in ft_model.wv:
            vec += ft_model.wv[word]
            count += 1
    if count > 0:
        vec /= count
    return vec

X = np.array([sent_vector(tokens) for tokens in df["tokens"]])

In [22]:
X.shape

(72000, 150)

In [27]:
le = LabelEncoder()
y = le.fit_transform(df["status"])

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=23)

In [29]:
xg_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    random_state=42
)

In [30]:
xg_model.fit(X_train, y_train)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [33]:
y_pred = xg_model.predict(X_test)

In [34]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7196527777777778


In [36]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

     Anxiety       0.84      0.83      0.83      3660
  Depression       0.68      0.70      0.69      3588
      Normal       0.76      0.76      0.76      3646
    Suicidal       0.60      0.58      0.59      3506

    accuracy                           0.72     14400
   macro avg       0.72      0.72      0.72     14400
weighted avg       0.72      0.72      0.72     14400



In [37]:
def predict_text(text):
    tokens = clean_text(text)
    vec = sent_vector(tokens)          # same FastText vector function
    vec = vec.reshape(1, -1)          # convert to 2D for model
    
    pred = xg_model.predict(vec)         # XGBoost prediction
    label = le.inverse_transform(pred)
    
    print("Prediction:", label[0])

In [38]:
predict_text("why am i nervous")

Prediction: Anxiety


In [42]:
predict_text('Start your day with intention and a smile, because mood always affects your day. ï¸')

Prediction: Normal


In [43]:
predict_text('Literally nobody wants to be my friend I am going to overdose. Getting rejected')

Prediction: Suicidal


In [44]:
import joblib

joblib.dump(xg_model, r"D:\M_h_project\xg_model.pkl")
joblib.dump(le, r"D:\M_h_project\label_encoder.pkl")
ft_model.wv.save(r"D:\M_h_project\fasttext_vectors.kv")

In [14]:
import sys
print(sys.version)

3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
